In [ ]:
# Ensure project root on sys.path for `src` imports
import sys
from pathlib import Path

cwd = Path.cwd()
for base in [cwd, cwd.parent, cwd.parent.parent]:
    if (base / "src").exists():
        sys.path.insert(0, str(base))
        break


# 04 - Generation

Assemble a prompt with retrieved context and produce an answer (dummy offline generator by default).



In [ ]:
from src.retrieve import retrieve
from src.generate import generate_answer

query = "Where can I see Macbeth in Colorado this summer?"
snippets = retrieve(query, k=5)
answer = generate_answer(query, snippets)
print(answer)


# 04 – Generation

Assemble a simple answer from retrieved context (no external LLM required).


In [ ]:
from pathlib import Path

from src.ingest import load_raw_documents
from src.embed import train_tfidf, embed_query
from src.retrieve import retrieve_top_k
from src.generate import simple_generate

processed_dir = "data/processed"
raw_dir = "data/raw"
if not any(Path(processed_dir).glob("*.txt")):
    documents = load_raw_documents(raw_dir)
else:
    documents = [p.read_text(encoding="utf-8") for p in Path(processed_dir).glob("*.txt")]

vectorizer, matrix = train_tfidf(documents)
q = "Where can I see Macbeth in Colorado this summer?"
qv = embed_query(q, vectorizer)
results = retrieve_top_k(qv, matrix, documents, top_k=5)
print(simple_generate(q, results))
